# Ethereum Gas Price Prediction — COM4532

**LSTM-based next-day Ethereum gas price forecasting** with XGBoost / ARIMA / SMA-7 / Naive baselines and SHAP explainability.

This notebook is built to run on **Kaggle** (recommended), **Google Colab**, or **locally**.
It auto-detects the environment and locates the 5 Etherscan CSV files.

**How to run on Kaggle:** create/open a notebook → *Add Data* → upload the 5 Etherscan CSVs as a Dataset → *Run All*.
See the README for the exact CSV list and download links.


## 0. Environment detection & data-path resolution

In [ ]:
import os, glob

def detect_env():
    if os.path.isdir('/kaggle/input'):
        return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except Exception:
        return 'local'

ENV = detect_env()
print('Environment:', ENV)

if ENV == 'kaggle':
    SEARCH_DIRS = glob.glob('/kaggle/input/*') + ['/kaggle/input']
    OUTPUT_DIR = '/kaggle/working'
elif ENV == 'colab':
    SEARCH_DIRS = ['/content', '/content/data', '/content/drive/MyDrive', '/content/drive/MyDrive/data', '.']
    OUTPUT_DIR = 'outputs'
else:
    SEARCH_DIRS = ['.', './data', '..', '../data']
    OUTPUT_DIR = 'outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_csv(keywords):
    """Return the first CSV whose filename contains ANY keyword (case-insensitive)."""
    for d in SEARCH_DIRS:
        for path in sorted(glob.glob(os.path.join(d, '**', '*.csv'), recursive=True)):
            name = os.path.basename(path).lower()
            if any(k in name for k in keywords):
                return path
    raise FileNotFoundError(
        f'No CSV matching {keywords} under {SEARCH_DIRS}. '
        'Upload the 5 Etherscan CSV files (see README).')

PATHS = {
    'gas_price': find_csv(['gasprice', 'avggasprice', 'averagegasprice']),
    'gas_used' : find_csv(['gasused']),
    'gas_limit': find_csv(['gaslimit', 'avggaslimit']),
    'tx_count' : find_csv(['txgrowth', 'transaction', '-tx', 'dailytx']),
    'eth_price': find_csv(['etherprice', 'ethprice', 'ether-price']),
}
print('Output dir:', OUTPUT_DIR)
for k, v in PATHS.items():
    print(f'{k:>10}: {v}')

## 1. Imports

On Kaggle/Colab everything below is preinstalled. The first block only installs missing packages when running locally.

In [ ]:
import importlib, subprocess, sys

def _ensure(module, pip_name=None):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or module], check=False)

for _m, _p in [('xgboost', 'xgboost'), ('shap', 'shap'),
               ('statsmodels', 'statsmodels'), ('seaborn', 'seaborn')]:
    _ensure(_m, _p)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
import shap

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow', tf.__version__)

## 2. Load & merge the 5 CSV files

Etherscan chart exports have the columns `Date(UTC), UnixTimeStamp, Value`. We skip the header row and assign our own names.

In [ ]:
gas_price = pd.read_csv(PATHS['gas_price'], skiprows=1, names=['Date', 'UnixTs', 'Gas_Price_Gwei'])
gas_used  = pd.read_csv(PATHS['gas_used'],  skiprows=1, names=['Date', 'UnixTs', 'Gas_Used'])
gas_limit = pd.read_csv(PATHS['gas_limit'], skiprows=1, names=['Date', 'UnixTs', 'Gas_Limit'])
tx_count  = pd.read_csv(PATHS['tx_count'],  skiprows=1, names=['Date', 'UnixTs', 'Tx_Count'])
eth_price = pd.read_csv(PATHS['eth_price'], skiprows=1, names=['Date', 'UnixTs', 'ETH_Price_USD'])

for d in [gas_price, gas_used, gas_limit, tx_count, eth_price]:
    d['Date'] = pd.to_datetime(d['Date'], errors='coerce')
    d.dropna(subset=['Date'], inplace=True)
    d.set_index('Date', inplace=True)
    d.drop(columns=['UnixTs'], inplace=True)
    col = d.columns[0]
    d[col] = pd.to_numeric(d[col], errors='coerce')

# Date range: post-EIP-1559. END_DATE matches the proposal scope (~880 days).
# Set END_DATE = None to use all available data instead.
START_DATE = '2021-08-05'
END_DATE   = '2023-12-31'

# Merge all into one frame (outer join so no day is silently dropped), then
# forward-fill the handful of missing days (proposal Sec. 3.4) before filtering.
df = gas_price.join([gas_used, gas_limit, tx_count, eth_price], how='outer')
df = df.ffill().bfill()
df = df[df.index >= START_DATE]
if END_DATE is not None:
    df = df[df.index <= END_DATE]

# Etherscan exports the average gas price in Wei. Convert to Gwei if needed.
gp_med = df['Gas_Price_Gwei'].median()
if gp_med > 1e6:
    df['Gas_Price_Gwei'] = df['Gas_Price_Gwei'] / 1e9
    print(f'Converted gas price Wei -> Gwei (raw median was {gp_med:,.0f}).')

# Derived feature + engineered feature + next-day target
df['Gas_Utilization'] = df['Gas_Used'] / df['Gas_Limit']
# Engineered feature (proposal Sec. 3.1): day-over-day change in gas price
# (first difference) -> captures the network's tightening / loosening momentum.
df['Gas_Price_Delta'] = df['Gas_Price_Gwei'].diff().fillna(0.0)
# Temporal feature (proposal Sec. 3.1): day-of-week encoded cyclically as sin/cos
# so that Sunday (6) sits next to Monday (0) instead of being treated as the most
# distant day -> captures the documented weekly seasonality (lower weekend fees)
# without imposing a false ordinal jump between week boundaries.
_dow = df.index.dayofweek  # 0 = Monday ... 6 = Sunday
df['DOW_Sin'] = np.sin(2 * np.pi * _dow / 7)
df['DOW_Cos'] = np.cos(2 * np.pi * _dow / 7)
df['Target'] = df['Gas_Price_Gwei'].shift(-1)
df.dropna(inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range   : {df.index.min().date()} -> {df.index.max().date()}')
print(f'Median gas price (Gwei): {df["Gas_Price_Gwei"].median():.4f}')
df.tail(3)

## 3. Exploratory visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
axes[0].plot(df.index, df['Gas_Price_Gwei'], color='steelblue', linewidth=0.8)
axes[0].set_title('Daily Average Gas Price (Gwei)')
axes[1].plot(df.index, df['Gas_Utilization'], color='darkorange', linewidth=0.8)
axes[1].set_title('Network Utilization Ratio (Gas Used / Gas Limit)')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='EIP-1559 target (50%)')
axes[1].legend()
axes[2].plot(df.index, df['Tx_Count'], color='seagreen', linewidth=0.8)
axes[2].set_title('Daily Transaction Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_overview.png'), dpi=150)
plt.show()

## 4. Preprocessing & chronological split

Three proposal-specified steps happen here:
1. **Winsorization** at the 99th percentile (caps computed on the training rows only) so
   genuine spike days are *kept* but no longer dominate the MSE.
2. **Min-max scaling** of features and target, fit **only on the training portion** to
   avoid look-ahead data leakage.
3. **Chronological 80/10/10 split** with a 30-day sliding window.

In [ ]:
FEATURES = ['Gas_Price_Gwei', 'Gas_Used', 'Gas_Limit', 'Gas_Utilization',
            'Tx_Count', 'ETH_Price_USD', 'Gas_Price_Delta',  # 6 base + 1 engineered delta
            'DOW_Sin', 'DOW_Cos']  # + day-of-week cyclical encoding (proposal Sec. 3.1)
TARGET = 'Target'
WINDOW = 30  # lookback window (days)

n = len(df)
train_end = int(n * 0.80)
val_end   = int(n * 0.90)

# --- Winsorize outliers at the 99th percentile (proposal Sec. 3.4) ---
# Caps are computed on TRAIN rows only, then applied everywhere, so no test
# information leaks back. Winsorizing (capping) keeps the spike rows instead of
# deleting them, but stops a handful of 400+ Gwei days from dominating the MSE.
WINSOR_Q = 0.99
# Only the continuous magnitude features are winsorized; the cyclical day-of-week
# encodings are already bounded in [-1, 1], so clipping them would be meaningless.
WINSOR_FEATURES = ['Gas_Price_Gwei', 'Gas_Used', 'Gas_Limit', 'Gas_Utilization',
                   'Tx_Count', 'ETH_Price_USD', 'Gas_Price_Delta']
feat_caps = df[WINSOR_FEATURES].iloc[:train_end].quantile(WINSOR_Q)
df[WINSOR_FEATURES] = df[WINSOR_FEATURES].clip(upper=feat_caps, axis=1)
tgt_cap = df[TARGET].iloc[:train_end].quantile(WINSOR_Q)
df[TARGET] = df[TARGET].clip(upper=tgt_cap)

# Fit feature scaler ONLY on training rows, then transform everything (no leakage)
scaler = MinMaxScaler()
scaler.fit(df[FEATURES].iloc[:train_end])
X_all = scaler.transform(df[FEATURES])
y_all = df[TARGET].values

# Build 30-day sliding-window sequences
X_seq, y_seq = [], []
for i in range(WINDOW, len(X_all)):
    X_seq.append(X_all[i - WINDOW:i])
    y_seq.append(y_all[i])
X_seq = np.array(X_seq)   # (samples, 30, 7)
y_seq = np.array(y_seq)

# Adjust split indices for the window offset
tr = train_end - WINDOW
vl = val_end - WINDOW
X_train, y_train = X_seq[:tr],   y_seq[:tr]
X_val,   y_val   = X_seq[tr:vl], y_seq[tr:vl]
X_test,  y_test  = X_seq[vl:],   y_seq[vl:]   # y_* stay in raw Gwei for evaluation

# --- Target: min-max scaled to [0, 1] (proposal Sec. 2.3), fit on TRAIN only ---
# Winsorization above already tames the spikes, so a plain min-max target is
# enough for the LSTM to train without collapsing to a constant mean prediction.
ty_scaler = MinMaxScaler()
ty_scaler.fit(y_seq[:tr].reshape(-1, 1))
y_seq_s = ty_scaler.transform(y_seq.reshape(-1, 1)).flatten()
y_train_s, y_val_s, y_test_s = y_seq_s[:tr], y_seq_s[tr:vl], y_seq_s[vl:]

def inv_target(y_scaled):
    """Map a scaled prediction back to raw Gwei."""
    y_scaled = np.asarray(y_scaled, dtype=float).reshape(-1, 1)
    return ty_scaler.inverse_transform(y_scaled).flatten()

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

## 5. LSTM model (64 -> 32 units, Dropout 0.2) + lightweight grid search

A two-layer LSTM (64 -> 32) with Dropout 0.2, a Dense(16, ReLU) projection and a single
output unit, trained with Adam + MSE and early stopping. A **lightweight grid search** over
learning rate and batch size selects hyperparameters on the chronological validation set
(proposal Sec. 3.4).

In [ ]:
def build_lstm(lr):
    m = Sequential([
        Input(shape=(WINDOW, len(FEATURES))),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss='mse')
    return m

# --- Lightweight grid search over learning rate and batch size (proposal Sec. 3.4) ---
# Each combo is validated on the chronological hold-out validation set (early stopping
# on val_loss), which keeps evaluation strictly forward in time. We pick the combo with
# the lowest validation loss, then retrain that configuration for longer.
LR_GRID    = [1e-3, 5e-4]
BATCH_GRID = [16, 32]

best = {'val': np.inf, 'lr': LR_GRID[0], 'bs': BATCH_GRID[-1]}
for lr in LR_GRID:
    for bs in BATCH_GRID:
        tf.random.set_seed(42)
        m = build_lstm(lr)
        h = m.fit(X_train, y_train_s, validation_data=(X_val, y_val_s),
                  epochs=40, batch_size=bs,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=8,
                                           restore_best_weights=True)],
                  verbose=0)
        v = float(min(h.history['val_loss']))
        print(f'lr={lr:<7} batch={bs:<3} -> best val_loss={v:.5f}')
        if v < best['val']:
            best = {'val': v, 'lr': lr, 'bs': bs}

print(f"\nBest hyperparameters: lr={best['lr']}, batch_size={best['bs']} (val_loss={best['val']:.5f})")

# --- Final training with the selected hyperparameters ---
tf.random.set_seed(42)
model = build_lstm(best['lr'])
model.summary()
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
history = model.fit(
    X_train, y_train_s,
    validation_data=(X_val, y_val_s),
    epochs=100, batch_size=best['bs'],
    callbacks=[es], verbose=1,
)

plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.title('LSTM Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_loss.png'), dpi=150)
plt.show()

## 5b. Walk-forward validation

Proposal Sec. 4.2 specifies a **walk-forward** procedure that reflects real deployment:
train on all data up to a point, predict one step ahead, then expand the training window
and repeat. Retraining an LSTM after *every single* day is too expensive, so we use the
standard lightweight approximation — an **expanding window refit every `REFIT_EVERY` days**
— and report walk-forward MAE/RMSE alongside the single-split metrics in Sec. 7.

In [ ]:
# --- Walk-forward validation (proposal Sec. 4.2) ---
# Expanding training window; the model is refit every REFIT_EVERY steps (a lightweight
# approximation of step-wise retraining). Between refits the current model predicts one
# day ahead, then that day's actual value is appended to the training pool.
REFIT_EVERY = 20  # days between refits -> only a few retrainings over the test horizon

hist_X = np.concatenate([X_train, X_val], axis=0)
hist_y = np.concatenate([y_train_s, y_val_s], axis=0)
wf_pred, n_refits, wf_model = [], 0, None
for i in range(len(X_test)):
    if i % REFIT_EVERY == 0:
        tf.random.set_seed(42)
        wf_model = build_lstm(best['lr'])
        wf_model.fit(hist_X, hist_y, epochs=30, batch_size=best['bs'],
                     callbacks=[EarlyStopping(monitor='loss', patience=6,
                                              restore_best_weights=True)],
                     verbose=0)
        n_refits += 1
    wf_pred.append(wf_model.predict(X_test[i:i+1], verbose=0).flatten()[0])
    # grow the training pool with the just-seen day
    hist_X = np.concatenate([hist_X, X_test[i:i+1]], axis=0)
    hist_y = np.concatenate([hist_y, y_test_s[i:i+1]], axis=0)

wf_pred_g = inv_target(np.array(wf_pred))
wf_mae  = mean_absolute_error(y_test, wf_pred_g)
wf_rmse = np.sqrt(mean_squared_error(y_test, wf_pred_g))
print(f'Walk-forward: {n_refits} refits (expanding window, refit every {REFIT_EVERY} days)')
print(f'  MAE={wf_mae:.4f}  RMSE={wf_rmse:.4f}  (compare with single-split LSTM in Sec. 7)')

## 6. Baseline models — Naive, SMA-7, ARIMA, XGBoost

In [ ]:
# --- Naive: tomorrow's price = today's price ---
naive_pred = y_test[:-1]
naive_true = y_test[1:]

# --- SMA-7: mean of previous 7 actual gas prices ---
test_start_idx = vl + WINDOW
full_series = df['Gas_Price_Gwei'].values
sma_preds = np.array([
    full_series[test_start_idx + i - 7: test_start_idx + i].mean()
    for i in range(len(y_test))
])

# --- ARIMA on the raw training gas-price series ---
train_series = df['Gas_Price_Gwei'].values[:train_end]
try:
    arima_fit = ARIMA(train_series, order=(5, 1, 0)).fit()
except Exception as e:
    print('ARIMA(5,1,0) failed, falling back to (2,1,2):', e)
    arima_fit = ARIMA(train_series, order=(2, 1, 2)).fit()
arima_preds = np.asarray(arima_fit.forecast(steps=len(y_test)))

# --- XGBoost on flattened 30x7 windows ---
X_train_xgb = X_train.reshape(X_train.shape[0], -1)
X_test_xgb  = X_test.reshape(X_test.shape[0], -1)
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
xgb.fit(X_train_xgb, y_train_s)                 # trained on the scaled target
xgb_preds = inv_target(xgb.predict(X_test_xgb)) # back to raw Gwei
print('Baselines ready.')

## 7. Evaluation metrics — MAE, RMSE, MAPE, R²

In [ ]:
def evaluate(name, true, pred):
    true = np.asarray(true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    eps  = 1e-8  # guard against division by ~0 in MAPE
    mape = np.mean(np.abs((true - pred) / np.maximum(np.abs(true), eps))) * 100
    r2   = r2_score(true, pred)
    print(f'{name:<10} MAE={mae:10.4f}  RMSE={rmse:10.4f}  MAPE={mape:8.2f}%  R2={r2:8.4f}')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

lstm_preds = inv_target(model.predict(X_test).flatten())   # back to raw Gwei

header = f'{"Model":<10} {"MAE":>14} {"RMSE":>15} {"MAPE":>11} {"R2":>10}'
print(header)
print('-' * len(header))
results = [
    evaluate('LSTM',    y_test,     lstm_preds),
    evaluate('XGBoost', y_test,     xgb_preds),
    evaluate('ARIMA',   y_test,     arima_preds),
    evaluate('SMA-7',   y_test,     sma_preds),
    evaluate('Naive',   naive_true, naive_pred),
]
metrics_df = pd.DataFrame(results).set_index('Model')
metrics_df

## 8. Prediction vs. actual (test set)

In [ ]:
test_dates = df.index[vl + WINDOW:]
plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_test,     label='Actual',  color='black',      linewidth=1.2)
plt.plot(test_dates, lstm_preds, label='LSTM',    color='steelblue',  linewidth=1.0, alpha=0.85)
plt.plot(test_dates, xgb_preds,  label='XGBoost', color='darkorange', linewidth=1.0, alpha=0.75)
plt.plot(test_dates, sma_preds,  label='SMA-7',   color='seagreen',   linewidth=0.8, linestyle='--')
plt.title('Gas Price Prediction — Test Set'); plt.xlabel('Date'); plt.ylabel('Gas Price (Gwei)')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'predictions.png'), dpi=150)
plt.show()

## 9. SHAP explainability

Two complementary explanations:
1. **LSTM (DeepExplainer)** — the proposal's chosen interface (Sec. 3.3). On GPU the LSTM
   uses a CuDNN kernel that has no SHAP gradient, so we fall back **DeepExplainer ->
   GradientExplainer -> KernelExplainer** (the last is model-agnostic and always works).
2. **XGBoost (TreeExplainer)** — fast and reliable; a complementary tree-based view of
   which time steps and features matter most.

In [ ]:
feat_names = [f'{f}_d{d + 1}' for d in range(WINDOW) for f in FEATURES]

# --- 1) Reliable: SHAP on XGBoost ---
tree_expl = shap.TreeExplainer(xgb)
xgb_shap = tree_expl.shap_values(X_test_xgb)
plt.figure()
shap.summary_plot(xgb_shap, X_test_xgb, feature_names=feat_names, max_display=15, show=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- 2) SHAP on the LSTM (proposal Sec. 3.3: DeepExplainer) ---
# DeepExplainer is the proposal's chosen interface. On some TF/Keras builds it fails
# (it patches TF's gradient registry), so we fall back to GradientExplainer and skip
# gracefully if neither works. SHAP is the last modeling step, so any gradient-registry
# side effects cannot affect earlier training cells.
n_bg = min(100, X_train.shape[0])
background = X_train[np.random.choice(X_train.shape[0], n_bg, replace=False)]
lstm_shap = None
try:
    sv = shap.DeepExplainer(model, background).shap_values(X_test[:50])
    lstm_shap = sv[0] if isinstance(sv, list) else sv
    print('LSTM SHAP computed via DeepExplainer')
except Exception as e1:
    print('DeepExplainer failed, falling back to GradientExplainer:', e1)
    try:
        sv = shap.GradientExplainer(model, background).shap_values(X_test[:50])
        lstm_shap = sv[0] if isinstance(sv, list) else sv
        print('LSTM SHAP computed via GradientExplainer')
    except Exception as e2:
        print('GradientExplainer also failed:', e2)

# --- 3) Model-agnostic fallback: KernelExplainer ---
# Deep/Gradient explainers differentiate the graph and break on GPU LSTMs (the CuDNN
# kernel 'CudnnRNNV3' has no SHAP gradient). KernelExplainer only calls model.predict,
# so it sidesteps the gradient registry entirely. It is slower, so we summarise the
# background with k-means and explain a small sample of the test set.
if lstm_shap is None:
    try:
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        bg = shap.kmeans(X_train_flat, 15)
        n_expl = min(20, X_test.shape[0])
        sample_flat = X_test[:n_expl].reshape(n_expl, -1)
        def _predict_flat(flat):
            seq = flat.reshape(flat.shape[0], WINDOW, len(FEATURES))
            return model.predict(seq, verbose=0).flatten()
        ke = shap.KernelExplainer(_predict_flat, bg)
        sv = ke.shap_values(sample_flat, nsamples=200)
        sv = sv[0] if isinstance(sv, list) else sv
        lstm_shap = np.asarray(sv).reshape(n_expl, WINDOW, len(FEATURES))
        print('LSTM SHAP computed via KernelExplainer (model-agnostic fallback)')
    except Exception as e3:
        print('KernelExplainer also failed:', e3)

if lstm_shap is not None:
    lstm_shap = np.asarray(lstm_shap)
    k = lstm_shap.shape[0]
    flat = lstm_shap.reshape(k, -1)
    plt.figure()
    shap.summary_plot(flat, X_test[:k].reshape(k, -1),
                      feature_names=feat_names, max_display=15, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'shap_lstm_summary.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('LSTM SHAP unavailable here; the XGBoost SHAP plot above serves as the explainability analysis.')

### SHAP interpretation

The two SHAP summaries answer the proposal's explainability question (Sec. 3.3) — *is a high
predicted gas price driven by transaction count, an ETH-price surge, or the recent base-fee
trajectory?*

- **Recent gas price dominates both models.** The top features are `Gas_Price_Gwei` at the most
  recent days of the 30-day window (`_d20`…`_d30`). For XGBoost the single most recent day
  (`Gas_Price_Gwei_d30`) is by far the strongest driver. This reflects the near **random-walk**
  nature of daily gas prices and explains why the Naïve baseline is so hard to beat.
- **ETH/USD price is the strongest secondary driver.** `ETH_Price_USD` appears high in both
  plots, supporting the proposal's hypothesis (Sec. 2.2) that ETH price acts as a market-sentiment
  proxy correlated with gas fees.
- **XGBoost uses a more diverse signal** (transaction count, gas limit, gas used) than the LSTM,
  which leans almost entirely on the recent price level — hence its smoother predictions.
- The engineered **`Gas_Price_Delta`** (Sec. 3.1) makes a small but non-zero contribution.

*Note:* in the KernelExplainer LSTM plot the SHAP values cluster on the negative side because of
the chosen base value (the k-means background mean); what matters for importance is the horizontal
**spread**, not the sign.

## 10. Save models & results

In [ ]:
import pickle

model.save(os.path.join(OUTPUT_DIR, 'lstm_gas_model.keras'))
with open(os.path.join(OUTPUT_DIR, 'xgb_gas_model.pkl'), 'wb') as f:
    pickle.dump(xgb, f)

results_df = pd.DataFrame({
    'Date':    test_dates,
    'Actual':  y_test,
    'LSTM':    lstm_preds,
    'XGBoost': xgb_preds,
    'ARIMA':   arima_preds,
    'SMA7':    sma_preds,
})
results_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions_all_models.csv'), index=False)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'metrics.csv'))
print('Saved to', OUTPUT_DIR, ':')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print('  ', f)